# TraceCore — Quick Start

**TraceCore** is a lightweight benchmark for action-oriented agents.  
It evaluates whether an agent can *operate*, not just reason — no LLM judges, no vibes.

This notebook walks through:
1. Installing TraceCore
2. Writing a minimal agent
3. Running it against a built-in task
4. Inspecting the result trace
5. Listing available tasks

> **Source & docs:** https://github.com/justindobbs/Tracecore

## 1. Install

In [1]:
%pip install "tracecore==0.9.3"

import importlib.metadata as _metadata
try:
    version = _metadata.version("tracecore")
    print(f"TraceCore installed ✓  v{version}")
except _metadata.PackageNotFoundError:
    print("TraceCore install complete, but version lookup failed.")

Note: you may need to restart the kernel to use updated packages.
TraceCore installed ✓  v0.9.3


## 2. Write a minimal agent

Every agent implements three methods:

| Method | Called when | Purpose |
|---|---|---|
| `reset(task_spec)` | Episode start | Receive task id, description, action schema, budgets |
| `observe(observation)` | Each step | Receive env state + last action result |
| `act() → dict` | Each step | Return `{"type": "<action>", "args": {...}}` |

The agent below solves **`filesystem_hidden_config`**: list a directory, read files, extract a config key, and submit the output.

The runner loads agents from a `.py` file by finding a class with `reset`, `observe`, and `act`.  
We write the class to a temp file, then pass the path to `run()`.

In [2]:
import pathlib, tempfile

AGENT_SRC = '''
class Agent:
    """Quickstart agent for filesystem_hidden_config."""

    def reset(self, task_spec):
        self.seen = set()
        self.obs = None

    def observe(self, observation):
        self.obs = observation

    def act(self):
        last_action = self.obs.get("last_action")
        last_result = self.obs.get("last_action_result")

        # Chain: read_file -> extract_value -> set_output
        if last_action and last_result and last_result.get("ok"):
            if last_action["type"] == "read_file":
                content = last_result.get("content", "")
                return {"type": "extract_value", "args": {"content": content, "key": "API_KEY"}}
            if last_action["type"] == "extract_value":
                value = last_result.get("value")
                if value:
                    return {"type": "set_output", "args": {"key": "API_KEY", "value": value}}

        # Read any newly discovered files
        files = self.obs.get("visible_state", {}).get("files_seen", [])
        for path in files:
            if path not in self.seen:
                self.seen.add(path)
                return {"type": "read_file", "args": {"path": path}}

        # Bootstrap: list the root directory
        return {"type": "list_dir", "args": {"path": "/app"}}
'''

agent_file = pathlib.Path(tempfile.mktemp(suffix="_agent.py"))
agent_file.write_text(AGENT_SRC)
print(f"Agent written to: {agent_file}")

Agent written to: C:\Users\justi\AppData\Local\Temp\tmpp5kxtdz8_agent.py


## 3. Run an episode

`run(agent_path, task_ref, seed)` is the single entry point.  
- `agent_path` — path to the `.py` file containing an agent class  
- `task_ref` — `"<task_id>@<version>"` (version is optional; defaults to latest)  
- `seed` — integer; same seed → identical episode (deterministic runtime)

The cell below runs the episode and prints a live update after every step, then shows the final summary.

In [3]:
import sys
from agent_bench.runner.runner import run

print("Running episode: filesystem_hidden_config@1  seed=42")
print("=" * 60)

result = run(str(agent_file), "filesystem_hidden_config@1", seed=42)

# Print a live update for every step in the trace
for entry in result["action_trace"]:
    step    = entry["step"]
    action  = entry["action"]
    res     = entry["result"]
    budget  = entry["budget_after_step"]
    ok      = res.get("ok", "?")
    status  = "ok" if ok is True else ("FAIL" if ok is False else str(ok))

    # Build a short detail string from the result
    detail = ""
    if ok is False:
        detail = f"  error={res.get('error', '?')}"
    elif action["type"] == "list_dir":
        detail = f"  found {len(res.get('files', []))} file(s)"
    elif action["type"] == "read_file":
        content = res.get("content", "")
        detail = f"  {len(content)} chars"
    elif action["type"] == "extract_value":
        detail = f"  value={res.get('value', '—')}"
    elif action["type"] == "set_output":
        detail = f"  key={action['args'].get('key')}"

    print(f"  step {step:>2}  {action['type']:<22}  [{status:<4}]{detail}")
    print(f"          budget → steps={budget['steps']}  tool_calls={budget['tool_calls']}")

print("=" * 60)
print(f"Result      : {'SUCCESS' if result['success'] else 'FAILED'}")
print(f"Termination : {result['termination_reason']}")
print(f"Steps used  : {result['steps_used']}")
print(f"Calls used  : {result['tool_calls_used']}")
if not result["success"]:
    print(f"Reason      : {result.get('failure_reason')}")
print(f"Run ID      : {result['run_id']}")

Running episode: filesystem_hidden_config@1  seed=42
  step  1  list_dir                [ok  ]  found 3 file(s)
          budget → steps=199  tool_calls=39
  step  2  read_file               [ok  ]  21 chars
          budget → steps=198  tool_calls=38
  step  3  extract_value           [ok  ]  value=correct_value
          budget → steps=197  tool_calls=37
  step  4  set_output              [ok  ]  key=API_KEY
          budget → steps=196  tool_calls=36
Result      : SUCCESS
Termination : success
Steps used  : 4
Calls used  : 4
Run ID      : ff02f4e288e04c11bf9ec272e8732cef


## 4. Available tasks

All built-in tasks are registered in `tasks/registry.json`.

In [4]:
from agent_bench.tasks.registry import list_task_descriptors

descriptors = list_task_descriptors()

print(f"{'Task ref':<38} {'Suite':<18} {'Steps':>5} {'Calls':>5}  Description")
print("-" * 110)
for desc in descriptors:
    budgets = desc.metadata.get("default_budget", {})
    steps = budgets.get("steps", "?")
    calls = budgets.get("tool_calls", "?")
    ref = f"{desc.id}@{desc.version}"
    print(f"{ref:<38} {desc.suite:<18} {steps:>5} {calls:>5}  {desc.description[:50]}")

Task ref                               Suite              Steps Calls  Description
--------------------------------------------------------------------------------------------------------------
deterministic_rate_service@1           api                  120    45  Survive a deterministic-but-demanding rate-limited
rate_limited_api@1                     api                  120    40  Retrieve a protected ACCESS_TOKEN from a mock HTTP
rate_limited_chain@1                   api                  140    60  Complete a multi-step handshake with a chained API
dice_game@1                            deterministic          3     3  Simple dice game for testing record mode. Agent mu
filesystem_hidden_config@1             filesystem           200    40  Extract the value of API_KEY from the filesystem.
config_drift_remediation@1             operations           180    50  Detect configuration drift and output the remediat
incident_recovery_chain@1              operations           170    45  Foll

## 5. Next steps

- **Scaffold an agent** — `agent-bench new-agent my_agent` (adds budget guards automatically)
- **Try other tasks** — swap `filesystem_hidden_config@1` for any ref listed above
- **Determinism check** — run the same agent + seed twice; results must be identical
- **Export a baseline** — `agent-bench baseline --agent my_agent.py --task <ref> --export latest`
- **Full docs** — https://github.com/justindobbs/Tracecore/tree/main/docs